## Notebook 03 — Data Profiling & Popularity / Imbalance Bias (MIND-small)

Goal of this notebook to try and be organized

- Load the processed click-level dataset created in Notebook 02.

- Produce basic profiling stats: dataset size, CTR, number of users/items.

- Measure how much political content exists in exposure and clicks.

- Quantify popularity concentration (how much exposure/clicks go to the top items).

- Save small summary tables that will be reused in later notebooks and in the report.

### Imports

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

### Path

In [2]:
PROJECT = Path().resolve().parent
DATA = PROJECT / "data"
PROCESSED = DATA / "processed"

print("PROJECT:", PROJECT)
print("PROCESSED exists?", PROCESSED.exists())
print("Files in processed:", [p.name for p in PROCESSED.iterdir()] if PROCESSED.exists() else "MISSING")

PROJECT: C:\Users\jlsmp\Documents\universidade\M.IA\IAS\project
PROCESSED exists? True
Files in processed: ['clicks_dev.pkl', 'clicks_train.pkl', 'news_dev.pkl', 'news_train.pkl']


### Load

Loading the click-level table (clicks_train.pkl) and the article table (news_train.pkl).

- clicks_train: one row per candidate article shown in an impression, with clicked = 0/1.

- news_train: one row per unique news article, with metadata like category/subcategory/title/abstract.

In [3]:
clicks = pd.read_pickle(PROCESSED / "clicks_train.pkl")
news = pd.read_pickle(PROCESSED / "news_train.pkl")

print("clicks:", clicks.shape)
print("news:", news.shape)

display(clicks.head(3))
display(news.head(3))


clicks: (5843444, 9)
news: (51282, 8)


,impression_id,user_id,time,history,news_id,clicked,category,subcategory,title
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689,1,sports,football_nfl,"Charles Rogers, former Michigan State football..."
1,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N35729,0,news,newsus,Porsche launches into second story of New Jers...
2,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678,0,sports,more_sports,Bode Miller delivered his twin boys after midw...


,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."


### Dataset profiling

In [4]:
total_rows = len(clicks)
total_clicks = int(clicks["clicked"].sum())
ctr = clicks["clicked"].mean()

n_users = clicks["user_id"].nunique()
n_items = clicks["news_id"].nunique()
n_impressions = clicks["impression_id"].nunique()

cands_per_imp = clicks.groupby("impression_id").size()
avg_cands = cands_per_imp.mean()
median_cands = cands_per_imp.median()

summary = pd.DataFrame([{
    "rows (candidates shown)": total_rows,
    "clicks": total_clicks,
    "CTR": ctr,
    "unique users": n_users,
    "unique items": n_items,
    "impressions": n_impressions,
    "avg candidates / impression": avg_cands,
    "median candidates / impression": median_cands,
}])

display(summary)


,rows (candidates shown),clicks,CTR,unique users,unique items,impressions,avg candidates / impression,median candidates / impression
0,5843444,236344,0.040446,50000,20288,156965,37.227688,24.0


- Total number of candidate impressions (rows in clicks)

- Total clicks and CTR (click-through rate)- this means how often a shown item gets clicked

- Unique users and unique items

- Average number of candidates per impression- on average, how many options were shown each time

### Politics and measure exposure/click share

MIND does not include an explicit publisher/source field for each article, and the URLs provided in news.tsv point to Microsoft-hosted pages rather than the original outlet domain. As a result, I could not directly group articles by publisher to study ideological exposure using outlet-level bias labels. To preserve the intended focus of the study (left/center/right polarization), I adopted a more advanced but systematic alternative that will be refered later.

For now I just define a simple politics vs non-politicis flag using the subcategory field.

This will let me:

- verify there is enough political content in the data

- maybe later study “reinforcement” effects (do political users receive more political recommendations?)

In [5]:
# politics = any subcategory containing "polit" (covers newspolitics, newsworldpolitics, etc.)
news["is_politics"] = news["subcategory"].str.contains("polit", case=False, na=False)

polit_ids = set(news.loc[news["is_politics"], "news_id"])
clicks["is_politics"] = clicks["news_id"].isin(polit_ids)

print("Unique political items:", news["is_politics"].sum())
print("Share of candidates shown that are politics:", clicks["is_politics"].mean())
print("Share of clicks that are politics:", clicks.loc[clicks["clicked"]==1, "is_politics"].mean())
print("Users who clicked politics:", clicks.loc[(clicks["clicked"]==1) & (clicks["is_politics"]), "user_id"].nunique())


Unique political items: 2831
Share of candidates shown that are politics: 0.04470206268768897
Share of clicks that are politics: 0.03825356260366246
Users who clicked politics: 6067


In [6]:
pol_clicks_per_user = (
    clicks.loc[(clicks["clicked"]==1) & (clicks["is_politics"])]
    .groupby("user_id").size()
)

print(pol_clicks_per_user.describe())
print("Users with >= 3 political clicks:", (pol_clicks_per_user >= 3).sum())
print("Users with >= 5 political clicks:", (pol_clicks_per_user >= 5).sum())


count    6067.000000
mean        1.490193
std         1.230731
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        27.000000
dtype: float64
Users with >= 3 political clicks: 627
Users with >= 5 political clicks: 173


Although many users have too few political clicks to reliably estimate user-level political preferences, the dataset still contains a substantial politically engaged subset (n=627 with ≥3 political clicks) and enough political content to support the intended analysis.